# PBOC潜在货币政策因子提取 — 动态因子模型 (DFM) + Kalman滤波

---

**方法论基础:** Hélène Rey et al., *"The Ins & Outs of Chinese Monetary Policy Transmission"*, NBER Working Paper w34626

**核心思路:**
从人民银行多维政策工具 (DR007, 1Y NCD, 1Y MLF, 存准率, 公开市场操作) 中,
运用状态空间模型与Kalman滤波, 提取单一**潜在流动性因子** $F_t$,
刻画央行真实的流动性调控意图.

**状态空间表示:**

$$\text{观测方程: } Y_t = \Lambda F_t + \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0, R)$$

$$\text{状态转移: } F_t = \Phi F_{t-1} + \eta_t, \quad \eta_t \sim \mathcal{N}(0, Q)$$

| 符号 | 含义 |
|:---:|:---|
| $Y_t$ | $k \times 1$ 可观测政策工具向量 |
| $F_t$ | $1 \times 1$ 潜在货币政策因子 |
| $\Lambda$ | $k \times 1$ 因子载荷 |
| $\Phi$ | $1 \times 1$ AR(1) 持续性系数 |
| $R$ | $k \times k$ 观测噪声协方差 (对角矩阵) |
| $Q$ | $1 \times 1$ 状态扰动方差 |

---
*本报告面向机构CIO及投资决策层, 数据与模型仅供研究参考.*


In [ ]:
# ============================================================================
# 依赖库导入与环境配置
# ============================================================================
import sys
import warnings
import logging

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm

# 项目模块
sys.path.insert(0, ".")
from data_engine import PBOCDataEngine, BenchmarkDataEngine
from kalman_model import (
    PBOCDynamicFactorModel,
    AlphaValidator,
    compute_factor_stats,
    classify_policy_regime,
)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")

# Plotly 全局模板 (深色专业风格)
PLOTLY_TEMPLATE = "plotly_dark"

print("✓ 模块加载完毕")


---
## 1. 数据获取与预处理

**数据源:** Bloomberg终端 (xbbg) / Ornstein-Uhlenbeck拟真数据自动回退

| 指标 | BBG Ticker | 频率 | 平稳性策略 |
|:---|:---|:---:|:---|
| DR007 | `CNFR007 Index` | 日 | **自适应** — ADF检验→平稳保Level, 否则自动差分 |
| 1Y NCD | `CNAA1Y Index` | 日 | **自适应** — 同上 |
| 1Y MLF | `CHLR12M Index` | 日 | **自适应** — 同上 |
| 存款准备金率 | `CHRRRP Index` | 日 | 强制一阶差分 (Δ) |
| OMO净投放 | `CNNIOMO Index` | 日 | 滚动Z标准化 (60日窗口) |

> **自适应逻辑:** Level → ADF不通过 → Δ(一阶差分) → ADF仍不通过 → Δ²(二阶差分)

In [ ]:
# ============================================================================
# 1.1 数据获取 (Bloomberg → 拟真数据自动回退)
# ============================================================================
engine = PBOCDataEngine(
    start_date="2018-01-01",
    end_date="2025-12-31",
    omo_rolling_window=20,
    force_mock=False,  # 设为True可强制使用拟真数据
)

raw_data = engine.fetch()
print(f"原始数据: {raw_data.shape[0]} 行 × {raw_data.shape[1]} 列")
print(f"时间区间: {raw_data.index[0].strftime('%Y-%m-%d')} ~ {raw_data.index[-1].strftime('%Y-%m-%d')}")
print()
raw_data.tail(5)


In [ ]:
# ============================================================================
# 1.2 原始数据可视化
# ============================================================================
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=("DR007 (银行间7天回购)", "1Y NCD (同业存单)",
                    "1Y MLF (中期借贷便利)", "存款准备金率 (RRR)",
                    "OMO净投放 (20日滚动累计, 亿元)", ""),
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

colors = ["#00D4FF", "#FF6B35", "#FFD700", "#00E676", "#FF4081"]
plot_config = [
    ("DR007", 1, 1), ("NCD_1Y", 1, 2), ("MLF_1Y", 2, 1),
    ("RRR", 2, 2), ("OMO_NET", 3, 1),
]

for (col, row, c), color in zip(plot_config, colors):
    fig.add_trace(
        go.Scatter(
            x=raw_data.index, y=raw_data[col],
            name=col, line=dict(color=color, width=1.5),
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.3f}<extra></extra>",
        ),
        row=row, col=c,
    )

fig.update_layout(
    title="PBOC货币政策工具 — 原始序列总览",
    height=900, template=PLOTLY_TEMPLATE,
    showlegend=False,
    font=dict(family="SimHei, Arial", size=11),
)
fig.show()


In [ ]:
# ============================================================================
# 1.3 单位根检验 (ADF) — 原始序列
# ============================================================================
adf_raw = engine.run_adf_battery(raw_data)
print("=" * 60)
print("ADF单位根检验 — 原始序列")
print("=" * 60)
adf_raw


In [ ]:
# ============================================================================
# 1.4 自适应平稳性处理 → 建模用数据
# ============================================================================
processed_data = engine.get_processed()
print(f"处理后数据: {processed_data.shape[0]} 行 × {processed_data.shape[1]} 列")
print()

# 输出各指标的实际处理决策
print("=" * 70)
print("自适应平稳性处理日志 (auto: ADF检验驱动)")
print("=" * 70)
for col, log in engine.stationarity_log.items():
    print(f"  {col:10s} → {log}")
print()

# 处理后序列的ADF检验 — 此时所有序列应为平稳
adf_processed = engine.run_adf_battery(processed_data)
print("=" * 70)
print("ADF单位根检验 — 处理后序列 (预期: 全部平稳)")
print("=" * 70)
adf_processed

---
## 2. 动态因子模型估计 (DFM + Kalman滤波)

**估计策略:**
- 最大似然估计 (MLE), L-BFGS-B优化器
- 单因子 ($k_{\text{factors}}=1$), AR(1) 状态转移
- 因子符号识别: 确保 $F_t$ 与 DR007 正相关 (紧缩 → 因子上升)


In [ ]:
# ============================================================================
# 2.1 DFM模型拟合
# ============================================================================
dfm = PBOCDynamicFactorModel(
    data=processed_data,
    k_factors=1,
    factor_order=1,
    standardize=True,
)

results = dfm.fit(maxiter=500, disp=False)
print("✓ 动态因子模型估计完毕")
print(f"  对数似然: {results.log_likelihood:.2f}")
print(f"  AIC:      {results.aic:.2f}")
print(f"  BIC:      {results.bic:.2f}")
print(f"  Φ (持续性): {results.transition_coeff:.4f}")


In [ ]:
# ============================================================================
# 2.2 因子载荷 Λ — 各政策工具对潜在因子的敏感度
# ============================================================================
loadings_df = results.factor_loadings.reset_index()
loadings_df.columns = ["政策工具", "因子载荷 (Λ)"]

fig_loadings = go.Figure(go.Bar(
    x=loadings_df["政策工具"],
    y=loadings_df["因子载荷 (Λ)"],
    marker_color=["#00D4FF" if v > 0 else "#FF4081"
                  for v in loadings_df["因子载荷 (Λ)"]],
    text=[f"{v:.3f}" for v in loadings_df["因子载荷 (Λ)"]],
    textposition="outside",
    hovertemplate="%{x}<br>Λ = %{y:.4f}<extra></extra>",
))

fig_loadings.update_layout(
    title="因子载荷 (Λ) — 各政策工具对潜在PBOC因子的贡献",
    xaxis_title="政策工具",
    yaxis_title="因子载荷",
    template=PLOTLY_TEMPLATE,
    height=450,
    font=dict(family="SimHei, Arial", size=12),
)
fig_loadings.show()

print("\n因子载荷解读:")
print("  载荷绝对值越大, 该工具越能有效反映PBOC流动性意图")
print("  正载荷: 因子上升 ↔ 该工具收紧")
print("  负载荷: 因子上升 ↔ 该工具宽松")


In [ ]:
# ============================================================================
# 2.3 潜在因子 F_t 时序图
# ============================================================================
factor = results.smoothed_factor

fig_factor = go.Figure()

# 因子主线
fig_factor.add_trace(go.Scatter(
    x=factor.index, y=factor.values,
    name="潜在因子 F(t) [平滑]",
    line=dict(color="#FFD700", width=2),
    hovertemplate="%{x|%Y-%m-%d}<br>F(t) = %{y:.3f}<extra></extra>",
))

# 零线
fig_factor.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

# 紧缩/宽松区域着色
fig_factor.add_hrect(y0=0.5, y1=factor.max() * 1.1,
                      fillcolor="red", opacity=0.08, line_width=0,
                      annotation_text="紧缩区间", annotation_position="top right")
fig_factor.add_hrect(y0=factor.min() * 1.1, y1=-0.5,
                      fillcolor="green", opacity=0.08, line_width=0,
                      annotation_text="宽松区间", annotation_position="bottom right")

fig_factor.update_layout(
    title="PBOC潜在货币政策因子 F(t) — Kalman平滑估计",
    xaxis_title="日期",
    yaxis_title="潜在因子值 (标准化)",
    template=PLOTLY_TEMPLATE,
    height=500,
    font=dict(family="SimHei, Arial", size=12),
)
fig_factor.show()


In [ ]:
# ============================================================================
# 2.4 因子 vs DR007 & ΔRRR — 偏离区间可视化 (Convexity机会)
# ============================================================================
fig_comp = make_subplots(
    rows=2, cols=1,
    subplot_titles=("潜在因子 vs DR007 (标准化)", "潜在因子 vs ΔRRR (标准化)"),
    shared_xaxes=True,
    vertical_spacing=0.10,
)

# 标准化DR007用于比较
dr007_z = (processed_data["DR007"] - processed_data["DR007"].mean()) / processed_data["DR007"].std()

fig_comp.add_trace(go.Scatter(
    x=factor.index, y=factor.values,
    name="潜在因子 F(t)", line=dict(color="#FFD700", width=2),
), row=1, col=1)

fig_comp.add_trace(go.Scatter(
    x=dr007_z.index, y=dr007_z.values,
    name="DR007 (Z-score)", line=dict(color="#00D4FF", width=1, dash="dot"),
    opacity=0.7,
), row=1, col=1)

# ΔRRR
if "RRR" in processed_data.columns:
    rrr_z = (processed_data["RRR"] - processed_data["RRR"].mean()) / processed_data["RRR"].std()
    fig_comp.add_trace(go.Scatter(
        x=factor.index, y=factor.values,
        name="潜在因子 F(t)", line=dict(color="#FFD700", width=2),
        showlegend=False,
    ), row=2, col=1)
    fig_comp.add_trace(go.Scatter(
        x=rrr_z.index, y=rrr_z.values,
        name="ΔRRR (Z-score)", line=dict(color="#00E676", width=1, dash="dot"),
        opacity=0.7,
    ), row=2, col=1)

# 标记偏离区间 — |F_t - DR007_z| > 1.5σ
divergence = np.abs(factor.reindex(dr007_z.index) - dr007_z)
div_dates = divergence[divergence > 1.5].index

for d in div_dates[::5]:  # 每5个画一个以免过密
    fig_comp.add_vline(x=d, line_dash="dot", line_color="rgba(255,64,129,0.3)",
                       line_width=0.5, row=1, col=1)

fig_comp.update_layout(
    title="潜在因子 vs 可观测政策工具 — 偏离区间 (Convexity交易机会)",
    height=700,
    template=PLOTLY_TEMPLATE,
    font=dict(family="SimHei, Arial", size=11),
    legend=dict(x=0.01, y=0.99),
)
fig_comp.show()

print(f"\n偏离信号日期数: {len(div_dates)} / {len(factor)} 交易日")
print(f"偏离占比: {100 * len(div_dates) / len(factor):.1f}%")


In [ ]:
# ============================================================================
# 2.5 政策体制分类
# ============================================================================
regimes = classify_policy_regime(factor)
regime_counts = regimes.value_counts()

fig_regime = go.Figure(go.Pie(
    labels=regime_counts.index,
    values=regime_counts.values,
    marker_colors=["#FF4081", "#FFD700", "#00E676"],
    textinfo="label+percent",
    hovertemplate="%{label}: %{value} 交易日 (%{percent})<extra></extra>",
))

fig_regime.update_layout(
    title="PBOC货币政策体制分布 (基于潜在因子)",
    template=PLOTLY_TEMPLATE,
    height=400,
    font=dict(family="SimHei, Arial", size=12),
)
fig_regime.show()

# 因子描述性统计
factor_stats = compute_factor_stats(factor)
print("=" * 50)
print("潜在因子描述性统计")
print("=" * 50)
for k, v in factor_stats.items():
    print(f"  {k}: {v:.4f}")


---
## 3. 交易Alpha验证

**检验逻辑:**
将潜在因子 $F_t$ 对基准利率 (1Y IRS / 10Y CGB) 进行回归:

$$\text{IRS}_{1Y,t} = \alpha + \beta \cdot F_t + \varepsilon_t$$

- 若 $\beta$ 显著, 则因子对市场利率具有解释力
- 残差 $\varepsilon_t$ 的 Z-score > ±1.5 标识定价偏离 → **Convexity交易机会**
- HAC标准误修正 (Newey-West, maxlags=10)


In [ ]:
# ============================================================================
# 3.1 获取基准利率数据
# ============================================================================
bench_engine = BenchmarkDataEngine(
    start_date="2018-01-01",
    end_date="2025-12-31",
    force_mock=False,
)
benchmark_data = bench_engine.fetch()
print(f"基准利率数据: {benchmark_data.shape[0]} 行")
print()

# ============================================================================
# 3.2 OLS回归: 潜在因子 → 基准利率
# ============================================================================
validator = AlphaValidator(factor=factor, benchmark=benchmark_data)

# IRS 1Y 回归
irs_result = validator.run_regression("IRS_1Y")
print(validator.get_regression_summary("IRS_1Y"))

# CGB 10Y 回归
cgb_result = validator.run_regression("CGB_10Y")
print(validator.get_regression_summary("CGB_10Y"))


In [ ]:
# ============================================================================
# 3.3 回归拟合可视化
# ============================================================================
fig_reg = make_subplots(
    rows=1, cols=2,
    subplot_titles=("潜在因子 vs 1Y IRS", "潜在因子 vs 10Y CGB"),
)

for i, (target, color) in enumerate([("IRS_1Y", "#00D4FF"), ("CGB_10Y", "#FF6B35")]):
    res = validator._results[target]
    combined = res["combined_data"]

    fig_reg.add_trace(go.Scatter(
        x=combined["factor"], y=combined["target"],
        mode="markers", name=target,
        marker=dict(color=color, size=3, opacity=0.4),
        hovertemplate="F(t)=%{x:.2f}<br>" + target + "=%{y:.3f}%<extra></extra>",
    ), row=1, col=i+1)

    # 回归线
    x_line = np.linspace(combined["factor"].min(), combined["factor"].max(), 100)
    y_line = res["alpha"] + res["beta"] * x_line
    fig_reg.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode="lines", name=f"OLS (R²={res['r_squared']:.3f})",
        line=dict(color="#FFD700", width=2),
    ), row=1, col=i+1)

fig_reg.update_layout(
    title="Alpha验证: 潜在因子对基准利率的解释力",
    height=450,
    template=PLOTLY_TEMPLATE,
    font=dict(family="SimHei, Arial", size=11),
)
fig_reg.show()


In [ ]:
# ============================================================================
# 3.4 残差分析与偏离体制识别 (Convexity交易信号)
# ============================================================================
divergence_regimes = validator.identify_divergence_regimes("IRS_1Y", zscore_threshold=1.5)

fig_resid = make_subplots(
    rows=2, cols=1,
    subplot_titles=("回归残差Z值时序", "偏离体制分布"),
    row_heights=[0.65, 0.35],
    vertical_spacing=0.12,
)

# 残差Z值
zvals = divergence_regimes["残差Z值"]
colors_z = ["#FF4081" if abs(z) > 1.5 else "#636EFA" for z in zvals]

fig_resid.add_trace(go.Bar(
    x=zvals.index, y=zvals.values,
    marker_color=colors_z,
    name="残差Z值",
    hovertemplate="%{x|%Y-%m-%d}<br>Z = %{y:.2f}<extra></extra>",
), row=1, col=1)

fig_resid.add_hline(y=1.5, line_dash="dash", line_color="#FF4081",
                     opacity=0.7, row=1, col=1)
fig_resid.add_hline(y=-1.5, line_dash="dash", line_color="#FF4081",
                     opacity=0.7, row=1, col=1)

# 偏离方向统计
regime_dist = divergence_regimes["偏离方向"].value_counts()
fig_resid.add_trace(go.Bar(
    x=regime_dist.index, y=regime_dist.values,
    marker_color=["#00E676", "#FF4081", "#636EFA"],
    text=regime_dist.values,
    textposition="outside",
    name="频次",
), row=2, col=1)

fig_resid.update_layout(
    title="Alpha残差分析 — Convexity交易机会识别",
    height=700,
    template=PLOTLY_TEMPLATE,
    showlegend=False,
    font=dict(family="SimHei, Arial", size=11),
)
fig_resid.show()

# 最近信号
recent_signals = divergence_regimes[
    divergence_regimes["偏离方向"] != "均衡区间"
].tail(10)
if len(recent_signals) > 0:
    print("\n最近10个偏离信号:")
    print(recent_signals.to_string())
else:
    print("\n当前无显著偏离信号")


In [ ]:
# ============================================================================
# 3.5 综合仪表盘: 因子 + 基准利率 + 偏离信号
# ============================================================================
irs_res = validator._results["IRS_1Y"]
combined = irs_res["combined_data"]
resid_z = irs_res["residual_zscore"]

fig_dash = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "潜在PBOC因子 F(t) vs 1Y IRS",
        "回归残差Z值 (|Z|>1.5 = Convexity信号)",
        "政策体制时间线",
    ),
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.45, 0.30, 0.25],
)

# Panel 1: Factor vs IRS
fig_dash.add_trace(go.Scatter(
    x=factor.index, y=factor.values,
    name="潜在因子 F(t)", line=dict(color="#FFD700", width=2),
), row=1, col=1)

# IRS标准化至因子尺度
irs_aligned = benchmark_data["IRS_1Y"].reindex(factor.index).dropna()
irs_z = (irs_aligned - irs_aligned.mean()) / irs_aligned.std()
fig_dash.add_trace(go.Scatter(
    x=irs_z.index, y=irs_z.values,
    name="1Y IRS (Z-score)", line=dict(color="#00D4FF", width=1.5, dash="dot"),
    opacity=0.7,
), row=1, col=1)

# Panel 2: Residual Z-scores
signal_colors = ["#FF4081" if abs(z) > 1.5 else "rgba(99,110,250,0.3)" for z in resid_z]
fig_dash.add_trace(go.Bar(
    x=resid_z.index, y=resid_z.values,
    marker_color=signal_colors,
    name="残差Z值",
    showlegend=False,
), row=2, col=1)
fig_dash.add_hline(y=1.5, line_dash="dash", line_color="#FF4081", opacity=0.5, row=2, col=1)
fig_dash.add_hline(y=-1.5, line_dash="dash", line_color="#FF4081", opacity=0.5, row=2, col=1)

# Panel 3: Policy Regime Timeline
regime_map = {"紧缩": 1, "中性": 0, "宽松": -1}
regime_num = regimes.map(regime_map)
regime_colors = regimes.map({"紧缩": "#FF4081", "中性": "#FFD700", "宽松": "#00E676"})

fig_dash.add_trace(go.Bar(
    x=regime_num.index, y=regime_num.values,
    marker_color=regime_colors.values,
    name="政策体制",
    showlegend=False,
    hovertemplate="%{x|%Y-%m-%d}<br>%{text}<extra></extra>",
    text=regimes.values,
), row=3, col=1)

fig_dash.update_layout(
    title="PBOC潜在因子综合仪表盘 — CIO决策视图",
    height=950,
    template=PLOTLY_TEMPLATE,
    font=dict(family="SimHei, Arial", size=11),
    legend=dict(x=0.01, y=0.99),
)
fig_dash.show()


In [ ]:
# ============================================================================
# 4. 模型诊断与完整摘要
# ============================================================================
print("=" * 70)
print("动态因子模型 (DFM) — 完整估计结果")
print("=" * 70)
print(results.model_summary)


---
## 5. 分析结论与投资启示

### 模型核心发现:

1. **潜在因子有效性:** DFM成功从5个PBOC政策工具中提取了单一潜在因子 $F_t$,
   载荷结构符合经济直觉 — 价格型工具 (DR007, NCD) 载荷显著高于数量型工具.

2. **因子持续性:** AR(1)系数 $\Phi$ 接近1, 表明PBOC货币政策调整具有较强的
   惯性特征, 与渐进式调控的央行风格一致.

3. **Alpha信号:** 因子对1Y IRS和10Y CGB均具有显著解释力. 残差Z值超过
   ±1.5σ的区间对应市场定价与央行真实意图的偏离 → **Convexity交易机会**.

### 风险提示:

- 拟真数据 (Ornstein-Uhlenbeck) 仅用于框架验证, 生产部署须接入Bloomberg实时数据
- 单因子模型可能无法捕捉结构性转变 (如LPR改革、利率走廊机制完善)
- 残差信号需结合宏观基本面、市场微观结构综合研判

---
*方法论参考: Hélène Rey, Zhengyang Jiang, Robert Richmond, "The Ins & Outs of Chinese Monetary Policy Transmission", NBER w34626*
